In [4]:
import os
import json
import random
import pickle  # Import pickle
import numpy as np

In [10]:

SOURCE_DATA_PATH = '/home/tomin/chorio/webapp_data/'
WEB_DATA_PATH = '/home/tomin/webapp/chorio-webgame/public/'

# Custom JSON encoder to handle NumPy arrays and other non-serializable types
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()  # Convert ndarray to list
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.bool_):
            return bool(obj)
        return super().default(obj)

def process_patient_data():
    """
    Processes patient data by:
    1. Creating metadata JSON files for control and regular patients
    2. Creating individual patient JSON files with their data respecting the original structure
    """
    # Create the patients directory if it doesn't exist
    patients_dir_path = os.path.join(WEB_DATA_PATH, "patients")
    if not os.path.exists(patients_dir_path):
        os.makedirs(patients_dir_path)
    
    # Process control patients
    control_folder_path = os.path.join(SOURCE_DATA_PATH, 'patient_profiles5_control')
    control_patients = process_patient_group(control_folder_path, True, patients_dir_path)
    
    # Save control patients metadata
    control_metadata_path = os.path.join(WEB_DATA_PATH, "control_patients_metadata.json")
    with open(control_metadata_path, "w") as f:
        json.dump(control_patients, f, indent=4, cls=NumpyEncoder)
    
    # Process regular patients
    regular_folder_path = os.path.join(SOURCE_DATA_PATH, 'patient_profiles5')
    regular_patients = process_patient_group(regular_folder_path, False, patients_dir_path)
    
    # Save regular patients metadata
    regular_metadata_path = os.path.join(WEB_DATA_PATH, "chorio_patients_metadata.json")
    with open(regular_metadata_path, "w") as f:
        json.dump(regular_patients, f, indent=4, cls=NumpyEncoder)
    
    print(f"Processed {len(control_patients)} control patients and {len(regular_patients)} chorio patients")
    print(f"Metadata saved to {control_metadata_path} and {regular_metadata_path}")
    print(f"Individual patient data saved to {patients_dir_path}")

def convert_numpy_objects(obj):
    """
    Recursively converts NumPy types to Python native types for JSON serialization
    """
    if isinstance(obj, dict):
        return {key: convert_numpy_objects(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_objects(item) for item in obj]
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.integer, np.int64, np.int32, np.int16, np.int8)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    else:
        return obj

def process_patient_group(folder_path, is_control, output_dir):
    """
    Process all patients in a group (control or regular)
    
    Args:
        folder_path: Path to the group folder
        is_control: Boolean indicating if these are control patients
        output_dir: Directory to save individual patient JSON files
        
    Returns:
        List of patient metadata dictionaries
    """
    patients_metadata = []
    
    # Get all patient folders (only folders with numeric names)
    try:
        patient_folders = [f for f in os.listdir(folder_path)
                          if os.path.isdir(os.path.join(folder_path, f)) and f.isdigit()]
    except FileNotFoundError:
        print(f"Warning: Folder not found: {folder_path}")
        return patients_metadata
    
    for patient_id in patient_folders:
        # Full path to patient folder
        patient_folder = os.path.join(folder_path, patient_id)
        
        # Path to the other_data subfolder
        other_data_folder = os.path.join(patient_folder, 'other_data')
        
        # Check if the expected structure exists
        if not os.path.exists(other_data_folder):
            print(f"Warning: other_data folder not found for patient {patient_id}")
            continue
        
        # Create patient data object
        patient_data = {
            "id": patient_id,
            "isControl": is_control,
        }
        
        # Load metadata.json from other_data folder
        metadata_path = os.path.join(other_data_folder, 'metadata.json')
        if os.path.exists(metadata_path):
            try:
                with open(metadata_path, 'r') as f:
                    metadata = json.load(f)
                patient_data["metadata"] = metadata
                
                # Add patient to metadata list
                patients_metadata.append({
                    "id": patient_id, 
                    "isControl": is_control,
                    "mrn": metadata.get("mrn", patient_id),
                    "class": metadata.get("class", None)
                })
            except Exception as e:
                print(f"Error loading metadata for patient {patient_id}: {e}")
                # Still add to metadata but with less info
                patients_metadata.append({
                    "id": patient_id,
                    "isControl": is_control
                })
        else:
            print(f"metadata.json not found for patient {patient_id}")
            # Add minimal info to metadata list
            patients_metadata.append({
                "id": patient_id,
                "isControl": is_control
            })
        
        # Load labs.pkl from other_data folder
        labs_path = os.path.join(other_data_folder, 'labs.pkl')
        if os.path.exists(labs_path):
            try:
                with open(labs_path, 'rb') as f:
                    labs_data = pickle.load(f)
                # Convert NumPy arrays to lists for JSON serialization
                labs_data = convert_numpy_objects(labs_data)
                patient_data["labs"] = labs_data
            except Exception as e:
                print(f"Error loading labs.pkl for patient {patient_id}: {e}")
        else:
            print(f"labs.pkl not found for patient {patient_id}")
        
        # Load vitals.pkl from other_data folder
        vitals_path = os.path.join(other_data_folder, 'vitals.pkl')
        if os.path.exists(vitals_path):
            try:
                with open(vitals_path, 'rb') as f:
                    vitals_data = pickle.load(f)
                # Convert NumPy arrays to lists for JSON serialization
                vitals_data = convert_numpy_objects(vitals_data)
                patient_data["vitals"] = vitals_data
            except Exception as e:
                print(f"Error loading vitals.pkl for patient {patient_id}: {e}")
        else:
            print(f"vitals.pkl not found for patient {patient_id}")
        
        # Save individual patient data to JSON file
        patient_json_path = os.path.join(output_dir, f"{patient_id}.json")
        try:
            with open(patient_json_path, "w") as f:
                json.dump(patient_data, f, indent=4, cls=NumpyEncoder)
            print(f"Successfully saved data for patient {patient_id}")
        except Exception as e:
            print(f"Error saving data for patient {patient_id}: {e}")
    
    return patients_metadata

process_patient_data()

vitals.pkl not found for patient 46798310
Successfully saved data for patient 46798310
vitals.pkl not found for patient 46875589
Successfully saved data for patient 46875589
vitals.pkl not found for patient 48003529
Successfully saved data for patient 48003529
vitals.pkl not found for patient 48966113
Successfully saved data for patient 48966113
vitals.pkl not found for patient 49468929
Successfully saved data for patient 49468929
vitals.pkl not found for patient 49743354
Successfully saved data for patient 49743354
vitals.pkl not found for patient 49893100
Successfully saved data for patient 49893100
vitals.pkl not found for patient 50266576
Successfully saved data for patient 50266576
Successfully saved data for patient 43038835
Successfully saved data for patient 46155941
Successfully saved data for patient 46440608
Successfully saved data for patient 46540829
Successfully saved data for patient 46708814
Successfully saved data for patient 46784591
Successfully saved data for patien

In [1]:
import torch
from torch.utils.data import Dataset
from torch import nn
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torch.autograd import Variable
import torch.optim as optim
from torch.utils.data import SequentialSampler


# import os
# import concurrent.futures

from torchinfo import summary

In [2]:
from model_components.models.exp66.model_data import IndividualDataQuery
from model_components.models.exp66.model_data import RiskScoreEncDec

In [5]:
cohort_source_dir = '/home/tomin/chorio/'


# open the pickle file in read binary mode
with open(cohort_source_dir+'cohort_datakeeper_full_new_3.pkl', 'rb') as f:
    # load the dictionary from the pickle file
    cohort_datakeeper3 = pickle.load(f)
with open(cohort_source_dir+'cohort_dataset_full_new_3.pkl', 'rb') as f:
    # load the dictionary from the pickle file
    cohort_dataset3 = pickle.load(f)
    
# open the pickle file in read binary mode
with open(cohort_source_dir+'cohort_datakeeper_full_new_10.pkl', 'rb') as f:
    # load the dictionary from the pickle file
    cohort_datakeeper = pickle.load(f)

with open(cohort_source_dir+'cohort_dataset_full_new_10.pkl', 'rb') as f:
    # load the dictionary from the pickle file
    cohort_dataset = pickle.load(f)
    
with open(cohort_source_dir+'cohort_dataset_full_new_8.pkl', 'rb') as f:
    # load the dictionary from the pickle file
    cohort_dataset8 = pickle.load(f)

cohort_dataset.admsn_context = cohort_dataset8.admsn_context


with open(cohort_source_dir+'cohort_dataset_vitals_mom_filtered_8.pickle', 'rb') as f:
    # load the dictionary from the pickle file
    cohort_datakeeper_vitals_mom_filtered = pickle.load(f)
    
with open(cohort_source_dir+'cohort_dataset_vitals_baby_filtered_8.pickle', 'rb') as f:
    # load the dictionary from the pickle file
    cohort_datakeeper_vitals_baby_filtered = pickle.load(f)
    
with open(cohort_source_dir+'cohort_dataset_labs_filtered_10.pickle', 'rb') as f:
    # load the dictionary from the pickle file
    cohort_datakeeper_labs_filtered = pickle.load(f)
    
with open(cohort_source_dir+'cohort_dataset_mom_labor_filtered_10.pickle', 'rb') as f:
    # load the dictionary from the pickle file
    cohort_datakeeper_mom_labor_filtered = pickle.load(f)
    
cohort_dataset.vitals_data_mom['chorio_dx=="Yes" and mom_age>=18'] = cohort_datakeeper_vitals_mom_filtered['chorio_dx=="Yes" and mom_age>=18']
cohort_dataset.vitals_data_mom['chorio_dx=="No" and mom_age>=18'] = cohort_datakeeper_vitals_mom_filtered['chorio_dx=="No" and mom_age>=18']
cohort_dataset.vitals_data_baby['chorio_dx=="Yes" and mom_age>=18'] = cohort_datakeeper_vitals_baby_filtered['chorio_dx=="Yes" and mom_age>=18']
cohort_dataset.vitals_data_baby['chorio_dx=="No" and mom_age>=18'] = cohort_datakeeper_vitals_baby_filtered['chorio_dx=="No" and mom_age>=18']
cohort_dataset.lab_data['chorio_dx=="Yes" and mom_age>=18'] = cohort_datakeeper_labs_filtered['chorio_dx=="Yes" and mom_age>=18']
cohort_dataset.lab_data['chorio_dx=="No" and mom_age>=18'] = cohort_datakeeper_labs_filtered['chorio_dx=="No" and mom_age>=18']
cohort_dataset.flo_labor_mom['chorio_dx=="Yes" and mom_age>=18'] = cohort_datakeeper_mom_labor_filtered['chorio_dx=="Yes" and mom_age>=18']
cohort_dataset.flo_labor_mom['chorio_dx=="No" and mom_age>=18'] = cohort_datakeeper_mom_labor_filtered['chorio_dx=="No" and mom_age>=18']

ModuleNotFoundError: No module named 'chorio4_src'